# AI 에스컬레이터 예방점검 스케줄러

과거 고장이력과 이용객 수를 분석해 고장 위험이 높은 에스컬레이터를 찾고, 담당자의 지역·경험·가능시간을 고려해 예방점검 일정을 추천하는 Multi-Agent 실습입니다.

> 이 실습은 정확한 고장 날짜를 예측하는 시스템이 아닙니다. 제한된 자료로 위험등급과 점검 우선순위를 추천하며, 실제 안전 판단과 일정 승인은 담당자가 수행합니다.

In [ ]:
# 최초 1회만 실행합니다.
# %pip install -U "crewai[openai]>=1.15,<2.0" python-dotenv pandas

In [7]:
import os
import warnings
from pathlib import Path
from dotenv import load_dotenv
from crewai import Agent, Crew, LLM, Process, Task

warnings.filterwarnings("ignore")

def find_env_file(start: Path) -> Path | None:
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    return None

env_path = find_env_file(Path.cwd())
if env_path is None:
    raise FileNotFoundError(".env 파일에 OPENAI_API_KEY를 설정하세요.")
load_dotenv(env_path, override=False)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY가 설정되지 않았습니다.")

model_name = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")
if not model_name.startswith("openai/"):
    model_name = f"openai/{model_name}"

llm = LLM(model=model_name, api_key=api_key, temperature=0.1)
print(f"사용 모델: {model_name}")

사용 모델: openai/gpt-4o-mini


## Agent 정의

각 Agent는 별도 학습모델이 아니라 `role`, `goal`, `backstory` 프롬프트로 서로 다른 업무를 수행합니다.

In [8]:
common_rules = (
    "제공된 데이터만 사용하고 없는 사실이나 수치를 만들지 않는다. "
    "정보가 부족하면 '추가 확인 필요'라고 표시한다. "
    "정확한 고장 날짜를 예측하지 않고 위험등급과 점검 필요성을 추천한다. "
    "결과는 한국어로 작성하며 실제 안전 판단과 일정 승인은 사람에게 남긴다."
)

machine_agent = Agent(
    role="기계 고장 분석 담당자",
    goal="고장이력, 부품, 설치연도와 환경정보만 사용해 자주 발생하거나 다시 발생하는 기계 고장과 점검 필요성을 찾는다.",
    backstory=(
        "에스컬레이터의 반복고장과 주변 환경을 함께 검토하는 기계 담당자다. "
        "이용객 수나 혼잡도는 보지 않고, 기계가 왜 점검을 필요로 하는지만 근거와 함께 판단한다. "
        "정확한 고장일을 맞히지 않으며 위험등급을 실제 고장확률이라고 표현하지 않는다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)


passenger_agent = Agent(
    role="지하철 이용객 이용패턴 분석 담당자",
    goal="역·요일·시간대별 이용객 흐름만 분석해 혼잡시간, 점검 비추천시간과 이용객이 적은 추천시간을 찾는다.",
    backstory=(
        "지하철 이용 흐름을 시간대별로 분석하는 담당자다. "
        "과거 지하철 이용객이 많은 시간적 특성을 고려하여 에스컬레이터 점검 비추천 및 추천 시간을 priority_agent에게 알려준다."
        "기계 고장이나 매출은 판단하지 않고, 언제 작업해야 이용객 불편이 적은지만 제시한다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

priority_agent = Agent(
    role="점검 종류·우선순위 결정 담당자",
    goal="기계 위험, 실시간 이상, 자체점검 기한과 계절·환경 조건을 종합해 긴급점검·수시특별점검·자체점검으로 분류하고 우선순위를 정한다.",
    backstory=(
        "기계 상태와 이용객 불편을 함께 검토해 점검 순서를 정하는 유지관리 책임자다. "
        "현재 운행 중지, 안전 관련 신호, 짧은 기간의 반복고장은 긴급점검으로 우선 검토한다. "
        "실시간 이상은 긴급점검, 계절·취약환경 대응은 수시특별점검, 소모품과 성능 유지는 자체점검으로 구분한다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

schedule_agent = Agent(
    role="현장 점검 일정 관리자",
    goal="긴급·수시특별·자체점검 목록과 이용패턴, 작업자 조건을 바탕으로 2인 1조·3교대·피로도를 고려한 현실적인 일정을 만든다.",
    backstory=(
        "현장 근무자의 지역, 경험, 교대시간과 피로도를 고려해 일정을 편성해 온 관리자다. "
        "긴급점검에는 같은 소속조의 2명을 배치하고 교대별 긴급대기조를 남겨둔다. 자체점검은 유지관리 자격을 확인하며 수시특별점검은 정해진 점검 주체를 표시한다. "
        "야간근무 다음 날 오전작업, 고난도 작업의 연속배정, 하루 최대작업 초과를 피한다. "
        "조직의 실제 운영규칙이 없으면 가정임을 표시하고 사람의 승인을 요청한다. "
        + common_rules
    ),
    llm=llm, allow_delegation=False, verbose=True,
)

## 샘플 데이터

실제 운영정보가 아닌 교육용 가상 데이터입니다. 시설·고장·이용객·담당자 정보만 사용합니다.

In [9]:
reference_date = "2026-07-24"

# data 디렉터리 CSV를 Agent 입력의 단일 원본으로 사용합니다.
import calendar
from datetime import date, timedelta
import pandas as pd
from IPython.display import display

def find_data_dir(start: Path) -> Path:
    for base in [start, *start.parents]:
        direct = base / "data"
        nested = base / "LLM모델이해" / "code" / "data"
        if (direct / "01_시설정보.csv").exists():
            return direct
        if (nested / "01_시설정보.csv").exists():
            return nested
    raise FileNotFoundError("data 디렉터리를 찾을 수 없습니다.")

data_dir = find_data_dir(Path.cwd())

def load_agent_csv(filename: str):
    dataframe = pd.read_csv(data_dir / filename, keep_default_na=False)
    return dataframe, dataframe.to_csv(index=False)

facility_df, facility_data = load_agent_csv("01_시설정보.csv")
failure_df, failure_data = load_agent_csv("02_고장이력.csv")
passenger_df, passenger_data = load_agent_csv("03_이용객패턴.csv")
staff_df, staff_data = load_agent_csv("04_작업자정보.csv")
special_df, special_inspection_data = load_agent_csv("05_수시특별점검기준.csv")
work_rule_df, work_rule_data = load_agent_csv("05_현장운영규칙.csv")
agent_column_map_df, _ = load_agent_csv("06_에이전트_칼럼_매핑.csv")

required_input_columns = {
    "시설정보": (facility_df, {"시설ID", "역명", "위치", "설치연도", "마지막점검일", "다음자체점검예정일", "현재상태", "실시간이상여부", "최근이상감지시각", "실내외", "최근환경", "대체이동수단"}),
    "고장이력": (failure_df, {"시설ID", "고장일", "고장종류", "운행중단시간_분", "조치내용"}),
    "이용객패턴": (passenger_df, {"역명", "요일구분", "시간대", "해당시간이용객", "혼잡수준", "환승역여부"}),
    "작업자정보": (staff_df, {"작업자ID", "담당자", "소속조", "교대", "가능지역", "경험분야", "보유자격", "가능일", "근무시간", "하루최대점검", "최근야간근무", "현재피로도", "현재상태"}),
    "수시특별점검기준": (special_df, {"점검명", "적용시기", "적용대상", "목적", "주요점검항목", "점검주체"}),
    "현장운영규칙": (work_rule_df, {"항목", "실습용운영규칙"}),
}
for data_name, (dataframe, required_columns) in required_input_columns.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{data_name} CSV 필수 칼럼 누락: {sorted(missing_columns)}")

# Agent마다 실제로 필요한 CSV 칼럼만 전달합니다.
machine_facility_columns = ["시설ID", "설치연도", "마지막점검일", "현재상태", "실시간이상여부", "최근이상감지시각", "실내외", "최근환경"]
passenger_facility_columns = ["시설ID", "역명", "위치", "대체이동수단"]
priority_facility_columns = ["시설ID", "역명", "다음자체점검예정일", "현재상태", "실시간이상여부", "최근이상감지시각", "실내외", "최근환경"]
machine_facility_data = facility_df[machine_facility_columns].to_csv(index=False)
passenger_facility_data = facility_df[passenger_facility_columns].to_csv(index=False)
priority_facility_data = facility_df[priority_facility_columns].to_csv(index=False)

reference_day = date.fromisoformat(reference_date)
schedule_month_start = (reference_day.replace(day=1) + timedelta(days=32)).replace(day=1)
schedule_month_end = schedule_month_start.replace(day=calendar.monthrange(schedule_month_start.year, schedule_month_start.month)[1])
schedule_month = schedule_month_start.strftime("%Y-%m")
schedule_start_date = schedule_month_start.isoformat()
schedule_end_date = schedule_month_end.isoformat()

print(f"데이터 경로: {data_dir.resolve()}")
print(f"시설 {len(facility_df)}건 / 고장이력 {len(failure_df)}건 / 이용객패턴 {len(passenger_df)}건 / 작업자 {len(staff_df)}명")
print(f"점검계획 기간: {schedule_start_date} ~ {schedule_end_date}")
display(staff_df.groupby(["교대", "소속조"]).size().rename("인원수").reset_index())
display(agent_column_map_df)

데이터 경로: /Users/chltmddn5843/Library/CloudStorage/GoogleDrive-chltmddn5843@gmail.com/내 드라이브/SKALA/2주차/LLM모델이해/code/data
시설 5건 / 고장이력 11건 / 이용객패턴 22건 / 작업자 35명
점검계획 기간: 2026-08-01 ~ 2026-08-31


,교대,소속조,인원수
0,대체,R01,1
1,석간,E01,2
2,석간,E02,2
3,석간,E03,2
4,석간,E04,2
5,석간,E05,2
6,석간,E06,2
7,야간,N01,2
8,야간,N02,2
9,야간,N03,2


,데이터파일,칼럼,설명,기계고장분석Agent,이용객패턴Agent,점검우선순위Agent,현장일정Agent,활용방식
0,01_시설정보.csv,시설ID,시설 고유 식별자,직접,직접,직접,간접,모든 결과 연결 기준
1,01_시설정보.csv,역명,시설이 있는 역,미사용,직접,직접,간접,이용객정보 연결 및 분류 결과 표시
2,01_시설정보.csv,위치,역 내부 설치 위치,미사용,직접,미사용,간접,환승통로·출입구 영향 및 현장 식별
3,01_시설정보.csv,설치연도,시설 사용기간,직접,미사용,미사용,미사용,오래된 시설의 보조 위험근거
4,01_시설정보.csv,마지막점검일,최근 점검일,직접,미사용,미사용,간접,점검 경과기간 판단
5,01_시설정보.csv,다음자체점검예정일,다음 자체점검 예정 날짜,미사용,미사용,직접,간접,자체점검 분류와 일정기한 판단
6,01_시설정보.csv,현재상태,정상운행·이상소음·점검대기,직접,미사용,직접,간접,긴급점검 분류의 핵심 근거
7,01_시설정보.csv,실시간이상여부,현재 실시간 이상 신호 여부,직접,미사용,직접,간접,긴급점검 분류와 즉시 인력배치 근거
8,01_시설정보.csv,최근이상감지시각,가장 최근 실시간 이상 신호 시각,직접,미사용,직접,간접,긴급점검 대응시점 판단
9,01_시설정보.csv,실내외,실내·실외 설치 구분,직접,미사용,직접,미사용,환경 노출 및 특별점검 대상 판단


## Task 정의

전문 Agent가 차례대로 분석하고 마지막 일정 관리자가 두 종류의 목록을 JSON으로 생성합니다.

In [10]:
machine_task = Task(
    description=(
        "기준일은 {reference_date}입니다. 다음 시설정보와 고장이력만 사용해 기계의 고장 위험을 분석하세요.\n\n"
        "[기계 판단용 시설정보]\n{machine_facility_data}\n[고장이력]\n{failure_data}\n\n"
        "시설정보의 시설ID·설치연도·마지막점검일·현재상태·실시간이상여부·최근이상감지시각·실내외·최근환경과 "
        "고장이력의 시설ID·고장일·고장종류·운행중단시간_분·조치내용만 사용하세요. "
        "전체 및 기준일 이전 최근 90일 고장횟수, 같은 고장 반복, 이전 조치 후 재발, 긴 운행중단과 마지막 점검 경과를 확인하세요. 환경은 원인으로 단정하지 말고 추가 확인 요인으로만 표시하세요. "
        "이용객 수와 혼잡은 판단하지 마세요. 각 시설에 기계위험(높음/중간/낮음), 비교점수(0~100), "
        "반복고장, 중점점검항목, 근거와 신뢰도를 목록으로 제시하세요."
    ),
    expected_output="이용객 정보가 배제된 시설별 기계위험·반복고장·점검항목·근거 목록",
    agent=machine_agent,
)

passenger_task = Task(
    description=(
        "다음 이용객 판단용 시설정보와 이용객패턴만 사용해 실제 이용패턴을 분석하세요.\n\n"
        "[이용객 판단용 시설정보]\n{passenger_facility_data}\n[이용객패턴]\n{passenger_data}\n\n"
        "시설정보의 시설ID·역명·위치·대체이동수단과 이용객패턴의 역명·요일구분·시간대·해당시간이용객·혼잡수준·환승역여부만 사용하세요. "
        "기계상태, 고장부품, 비용이나 매출은 판단하지 마세요. 각 시설에 이용객영향(매우 높음/높음/중간/낮음), "
        "영향 비교점수(0~100), 혼잡시간, 점검 비추천시간, 점검 추천시간과 근거를 제시하세요."
    ),
    expected_output="기계정보가 배제된 시설별 이용객영향·혼잡시간·점검 추천 및 비추천시간 목록",
    agent=passenger_agent,
)

priority_task = Task(
    description=(
        "기계 분석과 이용객 분석, 시설정보, 수시특별점검 기준을 합쳐 모든 시설을 긴급점검·수시특별점검·자체점검 중 하나로 분류하세요.\n\n"
        "[분류 판단용 시설정보]\n{priority_facility_data}\n[수시특별점검 기준]\n{special_inspection_data}\n\n"
        "시설정보의 시설ID·역명·다음자체점검예정일·현재상태·실시간이상여부·최근이상감지시각·실내외·최근환경과 특별점검 기준의 모든 칼럼만 직접 사용하세요. "
        "긴급점검: 실시간이상여부가 Y이거나 현재상태가 운행중지·점검대기이고, 앞선 기계 분석에서 최근 반복고장·재발이 확인되어 즉시 대응이 필요한 경우입니다. "
        "수시특별점검: 계획월 {schedule_month}이 적용시기에 해당하고 시설의 실내외·최근환경이 해빙기 또는 풍수해 기준과 일치하는 경우입니다. "
        "자체점검: 즉시 이상이나 계절 특별조건은 없지만 다음자체점검예정일이 지났거나 임박했거나, 윤활·브레이크·볼트·센서·소모품 확인이 필요한 경우입니다. "
        "여러 조건이 겹치면 긴급점검 > 수시특별점검 > 자체점검 순으로 한 종류만 선택하고, 선택되지 않은 조건은 추가 확인사항에 적으세요. "
        "기계위험을 가장 중요하게 보고 이용객영향은 같은 위험등급 안에서 순서를 조정하는 데 사용하세요. "
        "시설별 점검종류, 우선순위, 완료기한, 중점점검항목, 점검주체, 분류근거와 추가 확인사항을 작성하세요."
    ),
    expected_output="모든 시설의 긴급·수시특별·자체점검 단일 분류, 우선순위, 완료기한과 근거 목록",
    agent=priority_agent,
    context=[machine_task, passenger_task],
)

schedule_task = Task(
    description=(
        "점검 우선순위와 이용객 추천시간을 바탕으로 기준일 다음 달의 월간 현장 점검계획을 만드세요. 계획월은 {schedule_month}, 계획기간은 {schedule_start_date}부터 {schedule_end_date}까지입니다.\n\n"
        "[담당자정보]\n{staff_data}\n\n"
        "[현장 운영규칙]\n{work_rule_data}\n\n"
        "작업자정보의 작업자ID·담당자·소속조·교대·가능지역·경험분야·보유자격·가능일·근무시간·하루최대점검·최근야간근무·현재피로도·현재상태와 운영규칙만 사용하세요. "
        "긴급점검·수시특별점검·자체점검을 별도 목록으로 작성하고, 점검일과 운영일은 반드시 계획기간 안의 YYYY-MM-DD 실제 날짜로 작성하세요. 가능일의 요일과 날짜의 실제 요일이 일치해야 합니다. "
        "최근 야간근무와 현재피로도를 지키세요. 서로 다른 역 사이에는 30분 이동시간을 두고, "
        "긴급점검은 같은 소속조의 작업자 2명을 배정하세요. 자체점검은 승강기 유지관리 자격 보유자를 포함하고, 수시특별점검은 점검주체를 명시하세요. 이용객 혼잡시간은 긴급점검 외에는 피하세요. "
        "주간·석간·야간 각 교대별 최소 1개 조를 긴급대기 상태로 남기고, 배정할 수 없으면 '추가 인력 필요'라고 표시하세요. "
        "마지막으로 점검이 있는 각 운영일마다 주간·석간·야간 3개 행을 만들고, 배정점검조수와 실제 긴급대기조 코드를 제시하세요. 같은 조를 점검과 대기에 동시에 배정하지 마세요.\n\n"
        "반드시 설명이나 코드블록 없이 아래 구조의 유효한 JSON 하나만 출력하세요.\n"
        "{\"emergency_schedule\":[{\"우선순위\":1,\"점검일\":\"2026-08-03\",\"시간\":\"09:00\","
        "\"시설ID\":\"ES-01\",\"역명\":\"서울역\",\"담당조\":\"D01(김민수·강준호)\","
        "\"인원수\":2,\"중점점검항목\":\"발판·구동장치\",\"완료기한\":\"즉시\","
        "\"선정이유\":\"반복고장\",\"피로도확인\":\"적합\",\"승인상태\":\"담당자 확인 필요\"}],"
        "\"special_schedule\":[{\"우선순위\":1,\"점검일\":\"2026-08-05\",\"시간\":\"14:00\","
        "\"시설ID\":\"ES-02\",\"역명\":\"교대역\",\"담당조\":\"E01(조하린·장우진)\",\"인원수\":2,"
        "\"점검주체\":\"자체 관리 주체\",\"중점점검항목\":\"배수펌프·차수판·누전\",\"이용객고려\":\"비혼잡시간\",\"선정이유\":\"풍수해 대비\",\"피로도확인\":\"적합\",\"승인상태\":\"담당자 확인 필요\"}],"
        "\"self_schedule\":[{\"우선순위\":1,\"점검일\":\"2026-08-06\",\"시간\":\"10:00\","
        "\"시설ID\":\"ES-03\",\"역명\":\"용산역\",\"담당조\":\"D01(김민수·강준호)\",\"인원수\":2,"
        "\"자격확인\":\"승강기 유지관리 자격 포함\",\"중점점검항목\":\"윤활·브레이크·볼트·센서\",\"이용객고려\":\"비혼잡시간\",\"선정이유\":\"자체점검 기한\",\"피로도확인\":\"적합\",\"승인상태\":\"담당자 확인 필요\"}],"
        "\"staffing_plan\":[{\"계획월\":\"2026-08\",\"운영일\":\"2026-08-03\",\"교대\":\"주간\",\"배정점검조수\":2,"
        "\"긴급대기조\":\"D06\",\"필요인원\":6,\"판단근거\":\"점검 2개 조와 주간 대기 1개 조\","
        "\"확인사항\":\"실제 운영규칙 승인 필요\"},{\"계획월\":\"2026-08\",\"운영일\":\"2026-08-03\",\"교대\":\"석간\",\"배정점검조수\":0,\"긴급대기조\":\"E06\",\"필요인원\":2,\"판단근거\":\"석간 긴급대기\",\"확인사항\":\"담당자 승인 필요\"},{\"계획월\":\"2026-08\",\"운영일\":\"2026-08-03\",\"교대\":\"야간\",\"배정점검조수\":0,\"긴급대기조\":\"N05\",\"필요인원\":2,\"판단근거\":\"야간 긴급대기\",\"확인사항\":\"담당자 승인 필요\"}]}"
    ),
    expected_output="emergency_schedule, special_schedule, self_schedule, staffing_plan을 포함한 유효한 JSON 객체",
    agent=schedule_agent,
    context=[machine_task, passenger_task, priority_task],
)

## Crew 구성

In [11]:
crew = Crew(
    agents=[machine_agent, passenger_agent, priority_agent, schedule_agent],
    tasks=[machine_task, passenger_task, priority_task, schedule_task],
    process=Process.sequential,
    verbose=True,
)

## 실행 및 결과 저장

아래 셀은 API를 호출합니다. 긴급·수시특별·자체점검 일정과 인력운영안을 화면에 표시하고 CSV 파일로 저장합니다.

In [12]:
import json
import pandas as pd
from IPython.display import display

result = await crew.kickoff_async(
    inputs={
        "reference_date": reference_date,
        "machine_facility_data": machine_facility_data,
        "passenger_facility_data": passenger_facility_data,
        "priority_facility_data": priority_facility_data,
        "failure_data": failure_data,
        "passenger_data": passenger_data,
        "staff_data": staff_data,
        "special_inspection_data": special_inspection_data,
        "work_rule_data": work_rule_data,
        "schedule_month": schedule_month,
        "schedule_start_date": schedule_start_date,
        "schedule_end_date": schedule_end_date,
    }
)

raw_output = result.raw.strip()
if raw_output.startswith("```"):
    raw_output = raw_output.split("\n", 1)[1].rsplit("```", 1)[0].strip()
if raw_output.lower().startswith("json\n"):
    raw_output = raw_output[5:].strip()

try:
    final_data = json.loads(raw_output)
except json.JSONDecodeError as error:
    raise ValueError(f"최종 Agent가 유효한 JSON을 반환하지 않았습니다.\n{raw_output}") from error

required_sections = {"emergency_schedule", "special_schedule", "self_schedule", "staffing_plan"}
if not required_sections.issubset(final_data):
    raise ValueError(f"최종 결과 필수 목록 누락: {required_sections - set(final_data)}")

emergency_df = pd.DataFrame(final_data["emergency_schedule"])
special_df = pd.DataFrame(final_data["special_schedule"])
self_df = pd.DataFrame(final_data["self_schedule"])
staffing_df = pd.DataFrame(final_data["staffing_plan"])

emergency_columns = [
    "우선순위", "점검일", "시간", "시설ID", "역명", "담당조", "인원수",
    "중점점검항목", "완료기한", "선정이유", "피로도확인", "승인상태",
]
special_columns = [
    "우선순위", "점검일", "시간", "시설ID", "역명", "담당조", "인원수",
    "점검주체", "중점점검항목", "이용객고려", "선정이유", "피로도확인", "승인상태",
]
self_columns = [
    "우선순위", "점검일", "시간", "시설ID", "역명", "담당조", "인원수",
    "자격확인", "중점점검항목", "이용객고려", "선정이유", "피로도확인", "승인상태",
]
staffing_columns = [
    "계획월", "운영일", "교대", "배정점검조수", "긴급대기조", "필요인원",
    "판단근거", "확인사항",
]

missing_emergency = [c for c in emergency_columns if c not in emergency_df.columns]
missing_special = [c for c in special_columns if c not in special_df.columns]
missing_self = [c for c in self_columns if c not in self_df.columns]
missing_staffing = [c for c in staffing_columns if c not in staffing_df.columns]
if missing_emergency or missing_special or missing_self or missing_staffing:
    raise ValueError(
        f"필수 열 누락 - 긴급: {missing_emergency}, 수시특별: {missing_special}, 자체: {missing_self}, 인력: {missing_staffing}"
    )

emergency_df = emergency_df[emergency_columns].sort_values("우선순위").reset_index(drop=True)
special_df = special_df[special_columns].sort_values("우선순위").reset_index(drop=True)
self_df = self_df[self_columns].sort_values("우선순위").reset_index(drop=True)
staffing_df = staffing_df[staffing_columns].reset_index(drop=True)

# LLM 일정의 기본 조건을 코드로 한 번 더 확인합니다.
validation_messages = []
schedule_start_ts = pd.Timestamp(schedule_start_date)
schedule_end_ts = pd.Timestamp(schedule_end_date)
for schedule_name, schedule_df in [("긴급점검", emergency_df), ("수시특별점검", special_df), ("자체점검", self_df)]:
    parsed_dates = pd.to_datetime(schedule_df["점검일"], format="%Y-%m-%d", errors="coerce")
    for row_index, parsed_date in parsed_dates.items():
        if pd.isna(parsed_date):
            validation_messages.append(f"{schedule_name} {row_index + 1}행 점검일 형식 오류: YYYY-MM-DD 필요")
        elif not schedule_start_ts <= parsed_date <= schedule_end_ts:
            validation_messages.append(f"{schedule_name} {row_index + 1}행 점검일이 계획월({schedule_month}) 밖에 있음: {schedule_df.loc[row_index, '점검일']}")
staffing_dates = pd.to_datetime(staffing_df["운영일"], format="%Y-%m-%d", errors="coerce")
for row_index, parsed_date in staffing_dates.items():
    if pd.isna(parsed_date) or not schedule_start_ts <= parsed_date <= schedule_end_ts:
        validation_messages.append(f"인력운영 {row_index + 1}행 운영일 확인 필요: {staffing_df.loc[row_index, '운영일']}")
    if staffing_df.loc[row_index, "계획월"] != schedule_month:
        validation_messages.append(f"인력운영 {row_index + 1}행 계획월 불일치: {staffing_df.loc[row_index, '계획월']}")
required_shifts = {"주간", "석간", "야간"}
for operation_date, daily_staffing in staffing_df.groupby("운영일"):
    missing_shifts = required_shifts - set(daily_staffing["교대"])
    if missing_shifts:
        validation_messages.append(f"{operation_date} 인력운영 교대 누락: {sorted(missing_shifts)}")
    if daily_staffing["긴급대기조"].astype(str).str.strip().isin(["", "추가 인력 필요"]).any():
        validation_messages.append(f"{operation_date} 교대별 긴급대기조 확인 필요")

scheduled_ids = list(emergency_df["시설ID"]) + list(special_df["시설ID"]) + list(self_df["시설ID"])
expected_ids = set(facility_df["시설ID"])
missing_ids = expected_ids - set(scheduled_ids)
duplicate_ids = sorted({item for item in scheduled_ids if scheduled_ids.count(item) > 1})
if missing_ids:
    validation_messages.append(f"미배정 시설: {sorted(missing_ids)}")
if duplicate_ids:
    validation_messages.append(f"중복 배정 시설: {duplicate_ids}")

for _, row in emergency_df.iterrows():
    assigned_people = [name.strip() for name in str(row["담당조"]).split("·") if name.strip()]
    if int(row["인원수"]) != len(assigned_people):
        validation_messages.append(
            f"{row['시설ID']} 인원수 불일치: 담당조 {len(assigned_people)}명 / 기록 {row['인원수']}명"
        )
    if int(row["인원수"]) < 2:
        validation_messages.append(f"{row['시설ID']} 긴급점검 2인 1조 확인 필요")

if validation_messages:
    print("[일정 검증 경고]")
    for message in validation_messages:
        print("-", message)
else:
    print("기본 일정 검증 통과: 시설 누락·중복 및 긴급점검 인원 이상 없음")

emergency_style = (
    emergency_df.style
    .set_properties(subset=["우선순위", "완료기한"], **{"background-color": "#f8d7da", "font-weight": "bold"})
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_caption("긴급점검 일정")
)
special_style = (
    special_df.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_caption("수시특별점검 일정")
)
self_style = (
    self_df.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_caption("자체점검 일정")
)
staffing_style = (
    staffing_df.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_caption("교대·점검조·긴급대기조 제안")
)

display(emergency_style)
display(special_style)
display(self_style)
display(staffing_style)

output_dir = data_dir.parent
emergency_path = output_dir / "에스컬레이터_긴급점검_일정.csv"
special_path = output_dir / "에스컬레이터_수시특별점검_일정.csv"
self_path = output_dir / "에스컬레이터_자체점검_일정.csv"
staffing_path = output_dir / "에스컬레이터_인력운영_제안.csv"
emergency_df.to_csv(emergency_path, index=False, encoding="utf-8-sig")
special_df.to_csv(special_path, index=False, encoding="utf-8-sig")
self_df.to_csv(self_path, index=False, encoding="utf-8-sig")
staffing_df.to_csv(staffing_path, index=False, encoding="utf-8-sig")

print(f"긴급점검 CSV 저장 완료: {emergency_path.resolve()}")
print(f"수시특별점검 CSV 저장 완료: {special_path.resolve()}")
print(f"자체점검 CSV 저장 완료: {self_path.resolve()}")
print(f"인력운영 CSV 저장 완료: {staffing_path.resolve()}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97b20f69-a9a5-4081-a8e3-920bd62c411a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 기준일은 2026-07-24입니다. 다음 시설정보와 고장이력만 사용해 기계의 고장 위험을 분석하세요.              │
│                                                                                                                 │
│  [기계 판단용 시설정보]                                                                                         │
│  시설ID,설치연도,마지막점검일,현재상태,실시간이상여부,최근이상감지시각,실내외,최근환경                          │
│  ES-01,2013,2026-06-10,이상소음,Y,2026-07-24 08:10,실내,습도 높음                                               │
│  ES-02,2018,2026-06-25,정상운행,N,,실외,장마 노출                                                               │
│  ES-03,2021,2026-07-05,정상운행,N,,실내,특이사항 없음                                                           │
│  ES-04,2015,2026-05-30,점검대기,Y,2026-07-24 09:20,실외,고온·습도 높음                                          │
│  ES-05,2022,2026-07-10,정상운행,N,,실외,장마 노출                                                               │
│                                                                                                                 │
│  [고장이력]                                                                                                     │
│  시설ID,고장일,고장종류,운행중단시간_분,조치내용                                                                │
│  ES-01,2026-03-12,발판 이상,80,발판 조정                                                                        │
│  ES-01,2026-05-03,발판 이상,120,발판 부품 교체                                                                  │
│  ES-01,2026-06-15,발판 이상,90,임시 조정                                                                        │
│  ES-01,2026-07-18,이상 소음,40,점검 후 재가동                                                                   │
│  ES-02,2026-02-10,손잡이 속도 이상,30,벨트 조정                                                                 │
│  ES-02,2026-06-30,손잡이 속도 이상,50,벨트 조정                                                                 │
│  ES-03,2026-04-20,센서 이상,20,센서 청소                                                                        │
│  ES-04,2026-03-01,전원 이상,180,제어장치 점검                                                                   │
│  ES-04,2026-06-12,전원 이상,150,제어장치 재설정                                                                 │
│  ES-04,2026-07-20,운행 중 정지,210,원인 확인 중                                                                 │
│  ES-05,2026-01-15,조명 고장,0,조명 교체                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  시설정보의 시설ID·설치연도·마지막점검일·현재상태·실시간이상여부·최근이상감지시각·실내외·최근환경과 고장이력의  │
│  시설ID·고장일·고장종류·운행중단시간_분·조치내용만 사용하세요. 전체 및 기준일 이전 최근 90일 고장횟수, 같은     │
│  고장 반복, 이전 조치 후 재발, 긴 운행중단과 마지막 점검 경과를 확인하세요. 환경은 원인으로 단정하지 말고 추가  │
│  확인 요인으로만 표시하세요. 이용객 수와 혼잡은 판단하지 마세요. 각 시설에 기계위험(높음/중간/낮음),            │
│  비교점수(0~100), 반복고장, 중점점검항목, 근거와 신뢰도를 목록으로 제시하세요.                                  │
│  ID: 8c13abde-a3f8-4edb-99e0-512d4b25ca8b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 기계 고장 분석 담당자                                                                                   │
│                                                                                                                 │
│  Task: 기준일은 2026-07-24입니다. 다음 시설정보와 고장이력만 사용해 기계의 고장 위험을 분석하세요.              │
│                                                                                                                 │
│  [기계 판단용 시설정보]                                                                                         │
│  시설ID,설치연도,마지막점검일,현재상태,실시간이상여부,최근이상감지시각,실내외,최근환경                          │
│  ES-01,2013,2026-06-10,이상소음,Y,2026-07-24 08:10,실내,습도 높음                                               │
│  ES-02,2018,2026-06-25,정상운행,N,,실외,장마 노출                                                               │
│  ES-03,2021,2026-07-05,정상운행,N,,실내,특이사항 없음                                                           │
│  ES-04,2015,2026-05-30,점검대기,Y,2026-07-24 09:20,실외,고온·습도 높음                                          │
│  ES-05,2022,2026-07-10,정상운행,N,,실외,장마 노출                                                               │
│                                                                                                                 │
│  [고장이력]                                                                                                     │
│  시설ID,고장일,고장종류,운행중단시간_분,조치내용                                                                │
│  ES-01,2026-03-12,발판 이상,80,발판 조정                                                                        │
│  ES-01,2026-05-03,발판 이상,120,발판 부품 교체                                                                  │
│  ES-01,2026-06-15,발판 이상,90,임시 조정                                                                        │
│  ES-01,2026-07-18,이상 소음,40,점검 후 재가동                                                                   │
│  ES-02,2026-02-10,손잡이 속도 이상,30,벨트 조정                                                                 │
│  ES-02,2026-06-30,손잡이 속도 이상,50,벨트 조정                                                                 │
│  ES-03,2026-04-20,센서 이상,20,센서 청소                                                                        │
│  ES-04,2026-03-01,전원 이상,180,제어장치 점검                                                                   │
│  ES-04,2026-06-12,전원 이상,150,제어장치 재설정                                                                 │
│  ES-04,2026-07-20,운행 중 정지,210,원인 확인 중                                                                 │
│  ES-05,2026-01-15,조명 고장,0,조명 교체                                                                         │
│                                                                                                                 │
│                                                                                                                 │
│  시설정보의 시설ID·설치연도·마지막점검일·현재상태·실시간이상여부·최근이상감지시각·실내외·최근환경과 고장이력의  │
│  시설ID·고장일·고장종류·운행중단시간_분·조치내용만 사용하세요. 전체 및 기준일 이전 최근 90일 고장횟수, 같은     │
│  고장 반복, 이전 조치 후 재발, 긴 운행중단과 마지막 점검 경과를 확인하세요. 환경은 원인으로 단정하지 말고 추가  │
│  확인 요인으로만 표시하세요. 이용객 수와 혼잡은 판단하지 마세요. 각 시설에 기계위험(높음/중간/낮음),            │
│  비교점수(0~100), 반복고장, 중점점검항목, 근거와 신뢰도를 목록으로 제시하세요.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 기계 고장 분석 담당자                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### 시설별 기계 위험 분석                                                                                      │
│                                                                                                                 │
│  #### 1. 시설ID: ES-01                                                                                          │
│  - **기계위험**: 높음                                                                                           │
│  - **비교점수**: 85                                                                                             │
│  - **반복고장**: 발판 이상 (4회)                                                                                │
│  - **중점점검항목**: 발판 및 소음 관련 부품                                                                     │
│  - **근거**:                                                                                                    │
│    - 최근 90일 내 2회 발판 이상 고장 발생 (2026-03-12, 2026-05-03).                                             │
│    - 마지막 점검일(2026-06-10) 이후 이상 소음 발생 (2026-07-18).                                                │
│    - 고장 조치 후 재발 (발판 조정 및 부품 교체 후에도 반복).                                                    │
│  - **신뢰도**: 높음                                                                                             │
│                                                                                                                 │
│  #### 2. 시설ID: ES-02                                                                                          │
│  - **기계위험**: 중간                                                                                           │
│  - **비교점수**: 60                                                                                             │
│  - **반복고장**: 손잡이 속도 이상 (2회)                                                                         │
│  - **중점점검항목**: 벨트 및 손잡이 관련 부품                                                                   │
│  - **근거**:                                                                                                    │
│    - 최근 90일 내 2회 손잡이 속도 이상 고장 발생 (2026-02-10, 2026-06-30).                                      │
│    - 마지막 점검일(2026-06-25) 이후 정상운행 상태.                                                              │
│  - **신뢰도**: 중간                                                                                             │
│                                                                                                                 │
│  #### 3. 시설ID: ES-03                                                                                          │
│  - **기계위험**: 낮음                                                                                           │
│  - **비교점수**: 40                                                                                             │
│  - **반복고장**: 없음                                                                                           │
│  - **중점점검항목**: 센서 관련 부품                                                                             │
│  - **근거**:                                                                                                    │
│    - 최근 90일 내 고장 이력 없음 (2026-04-20 센서 이상은 단발성).                                               │
│    - 마지막 점검일(2026-07-05) 이후 정상운행 상태.                                                              │
│  - **신뢰도**: 높음                                                           

...

,우선순위,점검일,시간,시설ID,역명,담당조,인원수,점검주체,중점점검항목,이용객고려,선정이유,피로도확인,승인상태
0,1,2026-08-05,14:00,ES-02,교대역,E01(조하린·장우진),2,자체 관리 주체,벨트 및 손잡이 관련 부품,비혼잡시간,장마 노출로 인한 점검 필요,적합,담당자 확인 필요


,우선순위,점검일,시간,시설ID,역명,담당조,인원수,자격확인,중점점검항목,이용객고려,선정이유,피로도확인,승인상태
0,1,2026-08-06,10:00,ES-03,용산역,D02(이수진·윤서연),2,승강기 유지관리 자격 포함,센서 관련 부품,비혼잡시간,자체점검 기한,적합,담당자 확인 필요
1,2,2026-08-10,10:00,ES-05,구로역,D05(정하늘·서동혁),2,승강기 유지관리 자격 포함,조명 관련 부품,비혼잡시간,자체점검 기한,적합,담당자 확인 필요


,계획월,운영일,교대,배정점검조수,긴급대기조,필요인원,판단근거,확인사항
0,2026-08,2026-08-01,주간,2,D06,6,점검 2개 조와 주간 대기 1개 조,실제 운영규칙 승인 필요
1,2026-08,2026-08-02,주간,2,D06,6,점검 2개 조와 주간 대기 1개 조,실제 운영규칙 승인 필요
2,2026-08,2026-08-05,주간,2,E06,6,점검 1개 조와 주간 대기 1개 조,실제 운영규칙 승인 필요
3,2026-08,2026-08-06,주간,2,D06,6,점검 1개 조와 주간 대기 1개 조,실제 운영규칙 승인 필요
4,2026-08,2026-08-10,주간,2,D06,6,점검 1개 조와 주간 대기 1개 조,실제 운영규칙 승인 필요


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

긴급점검 CSV 저장 완료: /Users/chltmddn5843/Library/CloudStorage/GoogleDrive-chltmddn5843@gmail.com/내 드라이브/SKALA/2주차/LLM모델이해/code/에스컬레이터_긴급점검_일정.csv
수시특별점검 CSV 저장 완료: /Users/chltmddn5843/Library/CloudStorage/GoogleDrive-chltmddn5843@gmail.com/내 드라이브/SKALA/2주차/LLM모델이해/code/에스컬레이터_수시특별점검_일정.csv
자체점검 CSV 저장 완료: /Users/chltmddn5843/Library/CloudStorage/GoogleDrive-chltmddn5843@gmail.com/내 드라이브/SKALA/2주차/LLM모델이해/code/에스컬레이터_자체점검_일정.csv
인력운영 CSV 저장 완료: /Users/chltmddn5843/Library/CloudStorage/GoogleDrive-chltmddn5843@gmail.com/내 드라이브/SKALA/2주차/LLM모델이해/code/에스컬레이터_인력운영_제안.csv


## 테스트 체크리스트

실행 후 다음 항목을 확인합니다.

- 기계 고장 Agent가 이용객 데이터를 판단 근거로 사용하지 않는가?
- 이용객 Agent가 고장부품이나 매출을 판단하지 않고 혼잡·추천시간만 제시하는가?
- ES-01의 반복된 발판 이상과 ES-04의 최근 운행 중 정지가 긴급점검 후보에 반영되는가?
- 모든 시설이 긴급·수시특별·자체점검 중 하나로 분류되는가?
- 모든 점검일과 운영일이 기준일 다음 달인 2026-08-01~2026-08-31 안의 YYYY-MM-DD 날짜인가?
- Agent별 Task에서 언급한 판단 칼럼이 실제 전용 데이터 입력에 모두 존재하는가?
- 풍수해·해빙기 적용시기와 시설 환경이 수시특별점검 근거에 함께 제시되는가?
- 자체점검이 출퇴근 혼잡시간을 피하고 유지관리 자격자를 포함하는가?
- 긴급점검은 같은 소속조의 2인 1조로 배정되는가?
- 담당자의 지역·경험·교대·근무시간·피로도·하루 최대점검이 일정에 반영되는가?
- 다른 역 작업 사이에 이동시간 30분이 확보되는가?
- 주간·석간·야간 교대별 긴급대기조가 확보되는가?
- 출력의 모든 판단에 입력 데이터로 확인 가능한 근거가 있는가?